# Synthetic E-commerce Dataset and Benchmark Description

В данном ноутбуке представлено формальное описание synthetic `NL2SQL`-набора `synthetic_ecommerce`, использованного в локальном benchmark-сценарии. Цель описания состоит в фиксации структуры набора, принципов его построения и способа его использования при проверке качества генерации SQL-запросов.

Далее последовательно рассматриваются предметная область, реляционная схема, принципы формирования и наполнения `SQLite`-базы, состав SQL-запросов, распределение по уровням сложности, специально выделенные edge-case сценарии и порядок применения набора в execution-based оценке моделей. Такой формат позволяет явно зафиксировать, какие именно данные и запросы входят в benchmark-окружение и какие свойства этого окружения являются существенными для задачи `NL2SQL`.


## 1. Загружаемые артефакты

Набор `synthetic_ecommerce` представлен как согласованная совокупность исходных и производных артефактов. Такое разделение позволяет отдельно фиксировать схему данных, целевые SQL-запросы, заполненную базу данных и статистику покрытия, не смешивая эти уровни между собой.

В текущей версии используются следующие основные файлы:

- `dataset_v1.json` — базовое ядро набора, включающее описание схемы и 30 основных SQL-запросов;
- `edge_cases_v1.json` — дополнительный слой специальных кейсов для проверки пограничных SQL-сценариев;
- `seed.sql` — детерминированный сценарий заполнения `SQLite`-снимка;
- `snapshot.sqlite` — готовая база данных, используемая при выполнении запросов;
- `coverage_summary.json` — сводка по покрытию схемы, структуре запросов и их execution-характеристикам.

Наличие этих артефактов позволяет воспроизвести не только перечень примеров, но и само benchmark-окружение, в котором оценивается способность модели переходить от естественного языка к исполнимому SQL.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.options.display.max_colwidth = 200
pd.options.display.float_format = '{:.4f}'.format

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents)
    if (candidate / 'nl2sql').exists() and (candidate / 'shared').exists()
)

DATA_DIR = PROJECT_ROOT / 'data' / 'nl2sql' / 'synthetic_ecommerce'
DATASET_PATH = DATA_DIR / 'dataset_v1.json'
EDGE_PATH = DATA_DIR / 'edge_cases_v1.json'
SEED_SQL_PATH = DATA_DIR / 'seed.sql'
DB_PATH = DATA_DIR / 'snapshot.sqlite'
COVERAGE_PATH = DATA_DIR / 'coverage_summary.json'
BENCHMARK_SCRIPT_PATH = PROJECT_ROOT / 'benchmark_nl2sql.py'

with DATASET_PATH.open('r', encoding='utf-8') as handle:
    dataset = json.load(handle)
with EDGE_PATH.open('r', encoding='utf-8') as handle:
    edge_cases = json.load(handle)
with COVERAGE_PATH.open('r', encoding='utf-8') as handle:
    coverage = json.load(handle)

core_queries_df = pd.DataFrame(dataset['queries'])
edge_queries_df = pd.DataFrame(edge_cases['queries'])
schema_df = pd.DataFrame({'table_definition': dataset['schema']['tables']})
row_counts_df = pd.DataFrame(
    [{'table': key, 'rows': value} for key, value in coverage['row_counts'].items()]
).sort_values('table').reset_index(drop=True)
artifact_df = pd.DataFrame(
    [
        {'artifact': 'dataset_v1.json', 'purpose': 'core schema + 30 base queries', 'path': str(DATASET_PATH.relative_to(PROJECT_ROOT))},
        {'artifact': 'edge_cases_v1.json', 'purpose': 'supplemental edge-case query layer', 'path': str(EDGE_PATH.relative_to(PROJECT_ROOT))},
        {'artifact': 'seed.sql', 'purpose': 'deterministic SQLite data population', 'path': str(SEED_SQL_PATH.relative_to(PROJECT_ROOT))},
        {'artifact': 'snapshot.sqlite', 'purpose': 'execution target database', 'path': str(DB_PATH.relative_to(PROJECT_ROOT))},
        {'artifact': 'coverage_summary.json', 'purpose': 'schema/query coverage summary', 'path': str(COVERAGE_PATH.relative_to(PROJECT_ROOT))},
        {'artifact': 'benchmark_nl2sql.py', 'purpose': 'inference + execution evaluation entrypoint', 'path': str(BENCHMARK_SCRIPT_PATH.relative_to(PROJECT_ROOT))},
    ]
)

display(artifact_df)
display(row_counts_df)

,artifact,purpose,path
0,dataset_v1.json,core schema + 30 base queries,data/nl2sql/synthetic_ecommerce/dataset_v1.json
1,edge_cases_v1.json,supplemental edge-case query layer,data/nl2sql/synthetic_ecommerce/edge_cases_v1.json
2,seed.sql,deterministic SQLite data population,data/nl2sql/synthetic_ecommerce/seed.sql
3,snapshot.sqlite,execution target database,data/nl2sql/synthetic_ecommerce/snapshot.sqlite
4,coverage_summary.json,schema/query coverage summary,data/nl2sql/synthetic_ecommerce/coverage_summary.json
5,benchmark_nl2sql.py,inference + execution evaluation entrypoint,benchmark_nl2sql.py


,table,rows
0,categories,24
1,customers,420
2,order_items,1440
3,orders,720
4,products,220
5,returns,180


## 2. Предметная область и реляционная схема

Набор моделирует компактный e-commerce домен. Выбор именно такой предметной области обусловлен тем, что она сочетает интуитивно понятную бизнес-логику с достаточно типичным для `NL2SQL` набором сущностей и отношений: клиенты, товары, заказы, позиции заказа и возвраты. Благодаря этому схема остается содержательно прозрачной, но при этом позволяет формулировать запросы различной структурной сложности.

Схема включает шесть связанных таблиц:

- `customers` — сведения о клиентах и их регистрационных атрибутах;
- `categories` — товарные категории, включая иерархическую связь через `parent_category_id`;
- `products` — карточки товаров и их базовые характеристики;
- `orders` — заказы и связанные с ними платежные и логистические признаки;
- `order_items` — позиции внутри заказов;
- `returns` — информация о возвратах по отдельным строкам заказа.

Такая структура обеспечивает наличие как простых однотабличных запросов, так и многошаговых аналитических конструкций, включающих соединения, агрегацию, вложенные условия и проверку существования связанных записей. Иными словами, реляционная схема выбрана не произвольно, а как минимально достаточная модель прикладного процесса, на которой можно проверять основные типы `NL2SQL`-преобразований.


In [2]:
display(schema_df)

schema_summary_df = pd.DataFrame(
    [
        {'metric': 'Number of tables', 'value': len(dataset['schema']['tables'])},
        {'metric': 'Foreign keys enabled in snapshot build', 'value': True},
        {'metric': 'Database dialect', 'value': 'SQLite'},
    ]
)
display(schema_summary_df)

,table_definition
0,"customers(customer_id INT PRIMARY KEY, first_name VARCHAR, last_name VARCHAR, email VARCHAR, city VARCHAR, state VARCHAR, signup_date DATE, customer_segment VARCHAR)"
1,"categories(category_id INT PRIMARY KEY, category_name VARCHAR, parent_category_id INT, FOREIGN KEY (parent_category_id) REFERENCES categories(category_id))"
2,"products(product_id INT PRIMARY KEY, product_name VARCHAR, category_id INT, brand VARCHAR, unit_price DECIMAL(10,2), is_active BOOLEAN, launched_at DATE, FOREIGN KEY (category_id) REFERENCES categ..."
3,"orders(order_id INT PRIMARY KEY, customer_id INT, order_date TIMESTAMP, order_status VARCHAR, payment_method VARCHAR, shipping_country VARCHAR, shipping_state VARCHAR, total_amount DECIMAL(12,2), ..."
4,"order_items(order_item_id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price DECIMAL(10,2), discount_amount DECIMAL(10,2), FOREIGN KEY (order_id) REFERENCES orders(order_id), ..."
5,"returns(return_id INT PRIMARY KEY, order_item_id INT, return_date DATE, return_reason VARCHAR, refund_amount DECIMAL(10,2), return_status VARCHAR, FOREIGN KEY (order_item_id) REFERENCES order_item..."


,metric,value
0,Number of tables,6
1,Foreign keys enabled in snapshot build,True
2,Database dialect,SQLite


## 3. Принципы генерации и заполнения данных

Используемый `SQLite`-snapshot представляет собой не случайную выгрузку, а детерминированно сформированную benchmark-среду. Для задачи `NL2SQL` это принципиально важно, поскольку воспроизводимость результатов определяется не только фиксированным набором SQL-запросов, но и стабильностью самих данных, на которых эти запросы исполняются.

При построении снимка соблюдаются следующие принципы:

1. таблицы создаются непосредственно из описания схемы, зафиксированного в `dataset_v1.json`;
2. наполнение выполняется через `seed.sql`, что исключает неконтролируемые различия между запусками;
3. сохраняется согласованность внешних ключей между таблицами клиентов, заказов, позиций заказа, товаров и возвратов;
4. в данные намеренно включаются контролируемые специальные случаи, такие как `NULL`, пустые строки, редкие категории значений и отдельные нулевые величины.

В результате формируется база, сочетающая два необходимых свойства. С одной стороны, она остается похожей на прикладной набор данных предметной области e-commerce. С другой стороны, она обеспечивает стабильную execution-based валидацию, при которой различия в результатах моделей объясняются качеством SQL-генерации, а не изменчивостью исходных данных.


In [3]:
null_ratio_df = pd.DataFrame(
    [
        {'column': key, 'null_ratio': value}
        for key, value in coverage['data_quality_summary']['null_ratio_by_column'].items()
    ]
).sort_values(['null_ratio', 'column'], ascending=[False, True]).reset_index(drop=True)

unusual_values_df = pd.DataFrame(
    [
        {'check': key, 'count': value}
        for key, value in coverage['data_quality_summary']['unusual_value_checks'].items()
    ]
)

display(Markdown('**Размеры таблиц в готовом snapshot**'))
display(row_counts_df)

display(Markdown('**Контролируемые нестандартные значения**'))
display(unusual_values_df)

display(Markdown('**Колонки с ненулевой долей NULL**'))
display(null_ratio_df[null_ratio_df['null_ratio'] > 0].head(20))

**Размеры таблиц в готовом snapshot**

,table,rows
0,categories,24
1,customers,420
2,order_items,1440
3,orders,720
4,products,220
5,returns,180


**Контролируемые нестандартные значения**

,check,count
0,products.brand:Brand-X/Legacy,7
1,orders.payment_method:crypto,36
2,returns.return_reason:empty_string,14


**Колонки с ненулевой долей NULL**

,column,null_ratio
0,orders.shipping_state,0.3333
1,categories.parent_category_id,0.2500
2,returns.return_reason,0.1222
3,customers.state,0.1095
4,products.launched_at,0.0682
5,order_items.discount_amount,0.0382
6,customers.city,0.0262


## 4. Структура SQL-набора

SQL-часть набора организована двухслойно. Такое построение позволяет разделить основное benchmark-ядро, по которому рассчитываются основные метрики, и отдельный слой специальных проверок на пограничные случаи.

### 4.1. Базовое ядро

Основное ядро содержит `30` запросов и имеет сбалансированное распределение по сложности `10 / 10 / 10`. Такое деление используется для того, чтобы набор не смещался только в сторону простых фильтрующих запросов или, наоборот, только в сторону сложных аналитических конструкций.

- `easy` — запросы с однотабличной логикой, фильтрацией, диапазонами, сортировкой и ограничением результата;
- `medium` — запросы с соединениями, группировкой, базовой агрегацией и аналитическими срезами;
- `hard` — запросы с вложенными подзапросами, `HAVING`, `EXISTS / NOT EXISTS`, сравнением с агрегированными ориентирами и более сложной логикой отбора.

### 4.2. Supplemental edge-case слой

Дополнительный набор `edge_cases_v1.json` не смешивается с основным difficulty-ядром при расчёте основных метрик. Его назначение состоит в фиксации тех сценариев, которые важны для проверки устойчивости модели, но не должны искажать основной профиль сложности benchmark-набора. К таким сценариям относятся анти-соединения, обработка `NULL`, пустые результаты и отдельные агрегатные крайние случаи.


In [4]:
core_summary_df = pd.DataFrame(
    [
        {'metric': 'Core queries', 'value': coverage['query_counts']['core']},
        {'metric': 'Edge-case queries', 'value': coverage['query_counts']['edge_case']},
        {'metric': 'Total executable queries', 'value': coverage['query_counts']['total']},
    ]
)
difficulty_df = core_queries_df['difficulty'].value_counts().rename_axis('difficulty').reset_index(name='count')
edge_type_df = pd.DataFrame(
    [{'edge_type': key, 'count': value} for key, value in coverage['edge_type_distribution'].items()]
).sort_values('count', ascending=False)

display(core_summary_df)
display(difficulty_df)
display(edge_queries_df[['difficulty', 'group', 'edge_type', 'sql']])
display(edge_type_df)

,metric,value
0,Core queries,30
1,Edge-case queries,8
2,Total executable queries,38


,difficulty,count
0,easy,10
1,medium,10
2,hard,10


,difficulty,group,edge_type,sql
0,edge_case,edge_case,anti_join,"SELECT c.category_id, c.category_name FROM categories AS c LEFT JOIN products AS p ON p.category_id = c.category_id WHERE p.product_id IS NULL;"
1,edge_case,edge_case,anti_join,"SELECT c.customer_id, c.first_name, c.last_name FROM customers AS c LEFT JOIN orders AS o ON o.customer_id = c.customer_id WHERE o.order_id IS NULL;"
2,edge_case,edge_case,null_handling,"SELECT r.return_id, r.order_item_id, COALESCE(r.return_reason, 'unspecified') AS normalized_reason FROM returns AS r WHERE r.return_reason IS NULL OR r.return_reason = '';"
3,edge_case,edge_case,empty_result,"SELECT p.product_id, p.product_name FROM products AS p LEFT JOIN order_items AS oi ON oi.product_id = p.product_id WHERE oi.order_item_id IS NULL;"
4,edge_case,edge_case,null_handling,"SELECT o.shipping_country, COUNT(*) AS null_state_orders FROM orders AS o WHERE o.shipping_state IS NULL GROUP BY o.shipping_country;"
5,edge_case,edge_case,aggregation_edge,"SELECT c.customer_segment, AVG(o.total_amount) AS avg_cancelled_amount FROM customers AS c JOIN orders AS o ON o.customer_id = c.customer_id WHERE o.order_status = 'cancelled' GROUP BY c.customer_..."
6,edge_case,edge_case,aggregation_edge,"SELECT p.brand, COUNT(r.return_id) AS rejected_returns FROM products AS p JOIN order_items AS oi ON oi.product_id = p.product_id LEFT JOIN returns AS r ON r.order_item_id = oi.order_item_id AND r...."
7,edge_case,edge_case,empty_result,"SELECT c.customer_id, c.email FROM customers AS c WHERE EXISTS (SELECT 1 FROM orders AS o WHERE o.customer_id = c.customer_id AND o.total_amount = 0);"


,edge_type,count
0,aggregation_edge,2
1,anti_join,2
2,empty_result,2
3,null_handling,2


## 5. Какие именно SQL-паттерны покрываются

Набор сконструирован так, чтобы охватывать не один узкий класс запросов, а несколько устойчивых типов SQL-задач, характерных для прикладных аналитических и операционных сценариев. На структурном уровне он включает:

- однотабличные выборки, фильтрацию и сортировку;
- соединения `JOIN` между несколькими таблицами;
- агрегации и сегментацию через `GROUP BY`;
- сравнительные ограничения на агрегаты через `HAVING`;
- конструкции `EXISTS / NOT EXISTS`;
- обработку `NULL`, пустых строк и ситуаций с пустым результатом;
- арифметические выражения в агрегатах, возникающие на уровне строк заказа.

За счёт такого покрытия набор позволяет оценивать не только синтаксическую корректность предсказанного SQL, но и способность модели восстанавливать смысл исходного запроса: выбирать релевантные таблицы и поля, строить корректные соединения, использовать подходящие агрегаты и формулировать ограничения на нужном уровне детализации.


In [5]:
intent_df = pd.DataFrame(
    [{'intent': key, 'count': value} for key, value in coverage['intent_distribution'].items()]
).sort_values('count', ascending=False)
selectivity_df = pd.DataFrame(
    [{'selectivity': key, 'count': value} for key, value in coverage['selectivity_distribution'].items()]
).sort_values('count', ascending=False)
table_usage_df = pd.DataFrame(
    [{'table': key, 'usage_count': value} for key, value in coverage['table_usage'].items()]
).sort_values('usage_count', ascending=False)
column_usage_df = pd.DataFrame(
    [{'column': key, 'usage_count': value} for key, value in coverage['column_usage'].items()]
).sort_values('usage_count', ascending=False)

display(intent_df)
display(selectivity_df)
display(table_usage_df)
display(column_usage_df.head(20))

,intent,count
2,filtering,23
0,aggregation,21
4,segmentation,20
1,comparison,12
3,ranking,5


,selectivity,count
1,selective,26
0,non_selective,12


,table,usage_count
3,orders,20
2,order_items,17
4,products,15
1,customers,15
5,returns,8
0,categories,3


,column,usage_count
4,customers.customer_id,15
29,products.product_id,15
19,orders.order_id,14
17,orders.customer_id,13
14,order_items.product_id,11
13,order_items.order_item_id,8
24,orders.total_amount,8
32,returns.order_item_id,8
12,order_items.order_id,8
26,products.category_id,7


## 6. Как benchmark используется в пайплайне оценки

Synthetic e-commerce benchmark встроен в отдельный локальный пайплайн `benchmark_nl2sql.py`. По своей логике этот сценарий согласован с общим execution-based подходом домена `NL2SQL`, однако вместо внешних benchmark-датасетов использует локально контролируемую `SQLite`-базу и фиксированный набор запросов.

### 6.1. Входные данные

На вход benchmark-пайплайн получает:

- описание схемы и основного SQL-ядра из `dataset_v1.json`;
- готовый `SQLite`-snapshot;
- детерминированно формируемые `NL`-вопросы, построенные из шаблонов с явным grounding по таблицам, колонкам, соединениям и группировкам.

### 6.2. Режимы инференса

В benchmark-пайплайне поддерживаются два режима:

- `ea` — генерация одного SQL-кандидата при `temperature = 0`;
- `pass_k` — генерация нескольких независимых кандидатов для расчёта `pass@k`.

### 6.3. Принцип оценки

Оценка основана на реальном выполнении SQL на одной и той же `SQLite`-базе. Для каждого примера исполняется эталонный SQL и предсказанный SQL, после чего результаты нормализуются и сравниваются на уровне возвращаемых значений. Тем самым оценка ориентирована не на буквальное совпадение текстов запросов, а на совпадение их фактической семантики в терминах результата исполнения.

Такой способ проверки особенно важен для `NL2SQL`, поскольку один и тот же корректный ответ может быть выражен несколькими эквивалентными SQL-формулировками. Следовательно, benchmark измеряет прежде всего способность модели получать правильный результат на данных, а не воспроизводить заранее фиксированную строку запроса.


In [6]:
benchmark_contract_df = pd.DataFrame(
    [
        {'stage': 'Schema source', 'implementation': 'dataset_v1.json -> schema.tables'},
        {'stage': 'Execution database', 'implementation': 'snapshot.sqlite'},
        {'stage': 'NL generation', 'implementation': 'deterministic SQL-shaped templates with table/column grounding'},
        {'stage': 'Inference modes', 'implementation': 'ea and pass_k'},
        {'stage': 'Primary metric', 'implementation': 'execution match on SQLite result sets'},
        {'stage': 'Stored outputs', 'implementation': 'raw_predictions.jsonl, detailed_results.json, results.json'},
    ]
)

execution_profile_df = pd.DataFrame([coverage['result_distribution']])

display(benchmark_contract_df)
display(execution_profile_df)
display(pd.DataFrame(coverage['query_results']).head(12))

,stage,implementation
0,Schema source,dataset_v1.json -> schema.tables
1,Execution database,snapshot.sqlite
2,NL generation,deterministic SQL-shaped templates with table/column grounding
3,Inference modes,ea and pass_k
4,Primary metric,execution match on SQLite result sets
5,Stored outputs,"raw_predictions.jsonl, detailed_results.json, results.json"


,zero_rows_queries,one_row_queries,multi_row_queries,zero_or_one_ratio,is_skewed_to_zero_or_one
0,1,1,36,0.0526,False


,index,group,difficulty,edge_type,rows_returned,execution_time_ms,tables,intents,selectivity
0,1,core,easy,NaN,46,1.2293,[customers],[filtering],selective
1,2,core,easy,NaN,5,0.3370,[customers],[ranking],selective
2,3,core,easy,NaN,36,0.1798,[customers],[filtering],selective
3,4,core,easy,NaN,42,0.4145,[customers],[filtering],selective
4,5,core,easy,NaN,10,0.2398,[products],"[filtering, ranking]",selective
5,6,core,easy,NaN,6,0.1430,[products],[filtering],selective
6,7,core,easy,NaN,47,0.1641,[products],[filtering],selective
7,8,core,easy,NaN,10,0.2402,[orders],"[filtering, ranking]",selective
8,9,core,easy,NaN,379,0.7297,[orders],[filtering],selective
9,10,core,easy,NaN,30,0.0975,[orders],[filtering],selective


## 7. Что именно было сгенерировано: краткий формальный ответ

В терминах benchmark-дизайна был сформирован локальный synthetic `NL2SQL`-набор, включающий:

- фиксированную `SQLite`-схему e-commerce домена из шести связанных таблиц;
- детерминированно заполненный snapshot базы данных;
- ядро из 30 SQL-запросов, сбалансированное по уровням сложности `easy / medium / hard`;
- отдельный слой из 8 edge-case запросов;
- шаблонный механизм преобразования SQL в grounded `NL`-вопросы;
- execution-based benchmark-пайплайн для расчёта `EA`, `valid SQL rate` и `pass@k`.

Иными словами, в рамках данного сценария был сформирован не только перечень SQL-примеров, но и целостное воспроизводимое benchmark-окружение, в котором схема данных, наполнение базы, набор целевых запросов и процедура оценки согласованы между собой. Именно эта согласованность делает набор пригодным для контролируемой локальной проверки качества `NL2SQL`-моделей.
